# ¡Bienvenidos a la segunda parte de química cuántica!

<img src= "images/quantumjam-logo.svg" />

¡Hola de nuevo! En la primera parte armamos el problema. Vimos que la energía de una molécula es el autovalor más chico de su Hamiltoniano, y aprendimos a escribir ese Hamiltoniano como una suma de operadores de Pauli. En esta segunda parte vamos a resolverlo.

Vamos a ver dos algoritmos para calcular la energía del estado fundamental de una molécula con una computadora cuántica. El primero se llama VQE y es el más conocido de todos. El segundo es una idea más nueva, en la que la computadora cuántica solo propone configuraciones y una computadora clásica hace el resto del trabajo. Al final vamos a ver cómo evaluar un resultado cuando no tenemos la respuesta exacta para comparar.

**Instalación.** Para instalar todas las librerías necesarias, corran la siguiente celda apretando el botón de play a la izquierda.

In [ ]:
!pip install qiskit[visualization] qiskit-aer pyscf

import numpy as np
import matplotlib.pyplot as plt
from math import comb
from scipy.optimize import minimize
from pyscf import gto, scf, fci, mcscf, ao2mo, cc
from qiskit import QuantumCircuit, transpile
from qiskit.circuit import ParameterVector
from qiskit.circuit.library import PauliEvolutionGate, efficient_su2
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit.primitives import StatevectorEstimator
from qiskit_aer.primitives import EstimatorV2 as AerEstimator, SamplerV2 as AerSampler

La siguiente celda tiene las funciones que armamos en la primera parte, para no tener que copiarlas de nuevo. Si no recuerdan qué hace alguna, vuelvan a mirar la parte 1.

In [ ]:
def molecula(atomos, base="sto-3g", n_congelados=0):
    """Calcula Hartree-Fock, la energía exacta y las integrales de una molécula."""
    mol = gto.M(atom=atomos, basis=base, verbose=0)
    hf = scf.RHF(mol).run()

    # Los orbitales congelados no entran en el problema, ni tampoco sus electrones
    norb = mol.nao_nr() - n_congelados
    n_electrones = mol.nelectron - 2 * n_congelados

    # PySCF calcula las integrales h y g solo para los orbitales activos
    activo = mcscf.CASCI(hf, norb, n_electrones)
    h, constante = activo.get_h1cas()
    g = ao2mo.restore(1, activo.get_h2cas(), norb)

    return {
        "hf": hf,
        "norb": norb,
        "nelec": (n_electrones // 2, n_electrones // 2),
        "h": h,
        "g": g,
        "constante": constante,
        "e_hf": hf.e_tot,
        "e_fci": activo.kernel()[0],
    }


def destruccion_jordan_wigner(p, n_qubits):
    """Operador de destrucción del spin-orbital p con el mapeo de Jordan-Wigner."""
    # Z en los qubits 0, 1, ..., p-1 y (X + iY)/2 en el qubit p
    return SparsePauliOp.from_sparse_list(
        [("Z" * p + "X", list(range(p + 1)), 0.5),
         ("Z" * p + "Y", list(range(p + 1)), 0.5j)],
        n_qubits,
    )


def hamiltoniano_qubits(m, destruccion=destruccion_jordan_wigner):
    """Arma el Hamiltoniano de la molécula m como una suma de operadores de Pauli."""
    norb = m["norb"]
    n_qubits = 2 * norb
    h, g = m["h"], m["g"]

    # Operadores de destrucción y creación de cada spin-orbital
    # Los qubits 0, ..., norb-1 son los spin-orbitales alfa y los siguientes son los beta
    a = [destruccion(p, n_qubits) for p in range(n_qubits)]
    a_dag = [op.adjoint() for op in a]

    # salto[s][p][q] mueve un electrón con spin s del orbital q al orbital p
    salto = [[[a_dag[s * norb + p].dot(a[s * norb + q]).simplify()
               for q in range(norb)] for p in range(norb)] for s in range(2)]

    # Empezamos con la parte constante (repulsión entre núcleos y orbitales congelados)
    terminos = [SparsePauliOp("I" * n_qubits, m["constante"])]

    # Saltos de un electrón
    for s in range(2):
        for p in range(norb):
            for q in range(norb):
                terminos.append(h[p, q] * salto[s][p][q])

    # Saltos de dos electrones a la vez
    for s in range(2):
        for t in range(2):
            for p in range(norb):
                for q in range(norb):
                    segundo_salto = SparsePauliOp.sum(
                        [g[p, q, r, u] * salto[t][r][u] for r in range(norb) for u in range(norb)])
                    terminos.append(0.5 * salto[s][p][q].dot(segundo_salto))
                    if s == t:
                        # Corrección por el orden de los operadores cuando los dos electrones tienen el mismo spin
                        for u in range(norb):
                            terminos.append(-0.5 * g[p, q, q, u] * salto[s][p][u])

    return SparsePauliOp.sum(terminos).simplify(atol=1e-10)


def destruccion_paridad(p, n_qubits):
    """Operador de destrucción del spin-orbital p con el mapeo de paridad."""
    resto = list(range(p + 1, n_qubits))
    if p == 0:
        local = [("X", [0], 0.5), ("Y", [0], 0.5j)]
    else:
        local = [("ZX", [p - 1, p], 0.5), ("Y", [p], 0.5j)]
    # Además, X en todos los qubits que vienen después de p
    return SparsePauliOp.from_sparse_list(
        [(letras + "X" * len(resto), qubits + resto, coef) for letras, qubits, coef in local],
        n_qubits,
    )


def reducir_dos_qubits(H, m):
    """Elimina los dos qubits cuyo valor conocemos por las paridades de los electrones."""
    norb = m["norb"]
    n_alfa, n_beta = m["nelec"]
    # Qubit que guarda la paridad de los electrones alfa y qubit que guarda la paridad total
    qubits_conocidos = {norb - 1: (-1) ** n_alfa, 2 * norb - 1: (-1) ** (n_alfa + n_beta)}
    quedan = [k for k in range(2 * norb) if k not in qubits_conocidos]

    etiquetas, coeficientes = [], []
    for etiqueta, coef in H.to_list():
        letras = etiqueta[::-1]  # Damos vuelta la etiqueta para que letras[k] sea el qubit k
        for k, valor in qubits_conocidos.items():
            if letras[k] == "Z":
                coef = coef * valor  # Reemplazamos Z por su valor, +1 o -1
        etiquetas.append("".join(letras[k] for k in quedan)[::-1])
        coeficientes.append(coef)

    return SparsePauliOp(etiquetas, coeficientes).simplify()

# El principio variacional

Supongamos que tenemos un Hamiltoniano $\hat{H}$ cuya energía más baja es $E_0$. Si tomamos *cualquier* estado normalizado $\left|\psi\right>$ y calculamos su valor esperado, el resultado nunca va a ser menor que $E_0$.

\begin{equation}
\left<\psi\right|\hat{H}\left|\psi\right> \geq E_0
\end{equation}

Este resultado se conoce como **principio variacional**. Veamos por qué es cierto. Los autovectores $\left|E_k\right>$ de $\hat{H}$ forman una base ortonormal, así que podemos escribir cualquier estado como $\left|\psi\right> = \sum_k c_k\left|E_k\right>$, con $\sum_k |c_k|^2 = 1$. Entonces

\begin{equation}
\left<\psi\right|\hat{H}\left|\psi\right> = \sum_k |c_k|^2 E_k \geq \sum_k |c_k|^2 E_0 = E_0
\end{equation}

ya que todas las energías $E_k$ son mayores o iguales que $E_0$. Dicho en palabras, el valor esperado es un promedio de las energías posibles, y un promedio nunca puede ser menor que el valor más chico.

Esto nos da una estrategia. Si probamos muchos estados y nos quedamos con el de menor energía, nos vamos a ir acercando a $E_0$ desde arriba, sin pasarnos nunca. Cualquier energía que encontremos de esta forma es una **cota superior** de la energía exacta.

### Ejercicio 1

Consideren el Hamiltoniano de un qubit $\hat{H} = Z + 0{,}5\,X$.

a. Crea el Hamiltoniano con ```SparsePauliOp``` y calcula su autovalor más chico con ```np.linalg.eigvalsh()```.

b. Para 100 valores de $\theta$ entre $0$ y $2\pi$, prepara el estado $R_Y(\theta)\left|0\right>$ con un circuito y calcula el valor esperado de $\hat{H}$ (pueden usar ```Statevector``` y el método ```.expectation_value()```).

c. Grafica la energía en función de $\theta$, junto con una línea horizontal en el autovalor más chico. ¿La curva queda alguna vez por debajo de la línea? ¿Para qué valor de $\theta$ se alcanza el mínimo?

In [ ]:
# Escribe tu código aquí para la parte a

In [ ]:
# Escribe tu código aquí para la parte b

In [ ]:
# Escribe tu código aquí para la parte c

# VQE

## La idea

El **VQE** (por *Variational Quantum Eigensolver*, que se podría traducir como "resolvedor cuántico variacional de autovalores") lleva el ejercicio anterior a las moléculas. Fue propuesto en 2014 y se convirtió en uno de los algoritmos más estudiados para las computadoras cuánticas actuales, porque usa circuitos cortos y tolera algo de ruido.

El algoritmo combina una computadora cuántica y una computadora clásica.

<img src="images/quimica-ciclo-vqe.png" width="550" />

1. Elegimos un circuito con parámetros $\theta$, llamado **ansatz** (una palabra alemana que significa algo así como "punto de partida"). El circuito prepara un estado $\left|\psi(\theta)\right>$.
2. La computadora cuántica mide la energía $E(\theta) = \left<\psi(\theta)\right|\hat{H}\left|\psi(\theta)\right>$ con el ```Estimator```.
3. Un **optimizador** que corre en la computadora clásica mira la energía y propone nuevos parámetros que deberían dar una energía más baja.
4. Repetimos los pasos 2 y 3 hasta que la energía deja de bajar.

Por el principio variacional, la energía que obtenemos siempre es una cota superior de la energía exacta. Qué tan cerca lleguemos depende sobre todo de dos decisiones, qué ansatz usamos y qué optimizador usamos. Vamos a ver las dos.

La siguiente celda define la función ```vqe```, que hace todo el ciclo. Para el optimizador usa la función ```minimize``` de la librería SciPy, que recibe una función y busca los parámetros que la hacen lo más chica posible. Por ahora, vamos a usar el ```StatevectorEstimator``` de Qiskit, que calcula los valores esperados de forma exacta, como si pudiéramos medir infinitas veces sin ningún ruido. Más adelante vamos a agregar el ruido.

También definimos la función ```contar_cnots```, que nos va a servir para comparar circuitos. Las compuertas de dos qubits, como la CNOT, son las que más errores introducen en una computadora cuántica real, así que cuantas menos tenga un circuito, mejor.

In [ ]:
def vqe(ansatz, H, metodo="COBYLA", x0=None, estimator=None, maxiter=3000):
    """Busca los parámetros del ansatz que minimizan la energía de H."""
    if estimator is None:
        estimator = StatevectorEstimator()
    if x0 is None:
        x0 = np.zeros(ansatz.num_parameters)

    historial = []

    def energia(parametros):
        # La computadora cuántica (acá simulada) mide la energía para estos parámetros
        valor = float(estimator.run([(ansatz, H, parametros)]).result()[0].data.evs)
        historial.append(valor)
        return valor

    # El optimizador clásico propone parámetros nuevos hasta que la energía deja de bajar
    resultado = minimize(energia, x0, method=metodo, options={"maxiter": maxiter})
    return resultado, historial


def contar_cnots(circuito):
    """Cuenta las compuertas CNOT que quedan al transpilar el circuito."""
    transpilado = transpile(circuito, basis_gates=["cx", "rz", "sx", "x"], optimization_level=1)
    return transpilado.count_ops().get("cx", 0)

## Un primer ansatz

Una opción para el ansatz es usar un circuito genérico, que no sabe nada de química, armado con capas de rotaciones de un qubit y compuertas CNOT. Estos circuitos se llaman **ansatz eficientes para el hardware** (*hardware-efficient*, en inglés), porque usan compuertas que las computadoras cuánticas ejecutan con facilidad. Qiskit tiene uno listo, que se arma con la función ```efficient_su2```. El parámetro ```reps``` indica cuántas capas de CNOTs tiene.

In [ ]:
ansatz = efficient_su2(2, reps=1)
ansatz.decompose().draw("mpl")

### Ejercicio 2

a. Arma el Hamiltoniano de $H_2$ a 0,735 Å con el mapeo de paridad reducido (que tiene 2 qubits).

b. Crea el ansatz ```efficient_su2(2, reps=1)```. ¿Cuántos parámetros tiene? Pueden verlo con ```ansatz.num_parameters```.

c. Corre la función ```vqe``` con parámetros iniciales chicos y aleatorios. Pueden generarlos con ```np.random.default_rng(0).uniform(-0.1, 0.1, ansatz.num_parameters)```.

d. Grafica la energía en cada paso del optimizador (la lista ```historial```) junto con una línea horizontal en la energía exacta. Muestra el error final en mHa.

In [ ]:
# Escribe tu código aquí para la parte a

In [ ]:
# Escribe tu código aquí para la parte b

In [ ]:
# Escribe tu código aquí para la parte c

In [ ]:
# Escribe tu código aquí para la parte d

## Un ansatz químico

La otra opción es usar lo que sabemos de química para armar el ansatz. En la primera parte vimos que el estado exacto se parece bastante al estado de Hartree-Fock, y que las otras configuraciones aparecen cuando uno o dos electrones "saltan" de un orbital ocupado a uno vacío. Entonces, podemos armar el ansatz de esta forma.

1. Preparamos el estado de Hartree-Fock, aplicando una compuerta $X$ en cada spin-orbital ocupado.
2. Para cada salto posible de uno o de dos electrones, agregamos una compuerta que mezcla la configuración original con la configuración donde los electrones saltaron. Cada una de estas compuertas tiene un ángulo $\theta$ que controla cuánto se mezclan. Si todos los ángulos valen cero, el circuito no hace nada y nos quedamos con Hartree-Fock.

Si $T$ es el operador que hace saltar a los electrones (por ejemplo, $T = a_a^\dagger a_i$ para un electrón que salta del spin-orbital $i$ al $a$), la compuerta que mezcla las dos configuraciones es

\begin{equation}
e^{\theta\,(T - T^\dagger)}
\end{equation}

que es una compuerta unitaria. Qiskit puede armar compuertas de este tipo con la clase ```PauliEvolutionGate```, a partir de una suma de operadores de Pauli.

Este ansatz se llama **UCCSD** (por *unitary coupled cluster with singles and doubles*, en inglés), porque incluye los saltos de un electrón (*singles*) y de dos electrones (*doubles*). La siguiente celda lo arma para cualquier molécula, usando el mapeo de Jordan-Wigner. Al final, la función transpila el circuito una sola vez, porque si no Qiskit tendría que volver a armar todas las compuertas en cada paso del optimizador, y todo sería mucho más lento.

In [ ]:
def saltos_posibles(m):
    """Lista todos los saltos de uno y de dos electrones desde Hartree-Fock."""
    norb = m["norb"]
    n_alfa, n_beta = m["nelec"]
    # Spin-orbitales ocupados y vacíos en Hartree-Fock, para cada spin
    ocupados = [list(range(n_alfa)), [norb + i for i in range(n_beta)]]
    vacios = [list(range(n_alfa, norb)), [norb + i for i in range(n_beta, norb)]]

    saltos = []
    # Un electrón salta de i a a, sin cambiar su spin
    for s in range(2):
        for i in ocupados[s]:
            for a in vacios[s]:
                saltos.append(((i,), (a,)))
    # Dos electrones de distinto spin saltan a la vez
    for i in ocupados[0]:
        for j in ocupados[1]:
            for a in vacios[0]:
                for b in vacios[1]:
                    saltos.append(((i, j), (a, b)))
    # Dos electrones del mismo spin saltan a la vez
    for s in range(2):
        for x, i in enumerate(ocupados[s]):
            for j in ocupados[s][x + 1:]:
                for y, a in enumerate(vacios[s]):
                    for b in vacios[s][y + 1:]:
                        saltos.append(((i, j), (a, b)))
    return saltos


def ansatz_ucc(m):
    """Ansatz químico que parte de Hartree-Fock y agrega un ángulo por cada salto posible."""
    norb = m["norb"]
    n_alfa, n_beta = m["nelec"]
    n_qubits = 2 * norb
    a = [destruccion_jordan_wigner(p, n_qubits) for p in range(n_qubits)]
    saltos = saltos_posibles(m)
    theta = ParameterVector("θ", len(saltos))

    circuito = QuantumCircuit(n_qubits)
    # Estado de Hartree-Fock, con X en los spin-orbitales ocupados
    for i in range(n_alfa):
        circuito.x(i)
    for i in range(n_beta):
        circuito.x(norb + i)

    for angulo, (desde, hacia) in zip(theta, saltos):
        # T saca los electrones de "desde" y los pone en "hacia"
        T = SparsePauliOp("I" * n_qubits)
        for p in hacia:
            T = T.dot(a[p].adjoint())
        for p in reversed(desde):
            T = T.dot(a[p])
        # G = i(T - T†) es hermítico, y la compuerta aplica exp(θ(T - T†))
        G = (1j * (T - T.adjoint())).simplify()
        circuito.append(PauliEvolutionGate(G, time=angulo), range(n_qubits))

    # Transpilamos una sola vez para que la simulación sea rápida
    return transpile(circuito, basis_gates=["cx", "rz", "sx", "x"], optimization_level=1)

In [ ]:
m = molecula("H 0 0 0; H 0 0 0.735")
print("Saltos posibles =", saltos_posibles(m))

ansatz = ansatz_ucc(m)
print("Parámetros =", ansatz.num_parameters)
print("CNOTs =", contar_cnots(ansatz))

Cada salto está escrito como dos tuplas, con los spin-orbitales de donde salen los electrones y los spin-orbitales adonde llegan. Recuerden que los spin-orbitales 0 y 1 son los $\alpha$ y los spin-orbitales 2 y 3 son los $\beta$.

### Ejercicio 3

a. Crea el ansatz químico de $H_2$ a 0,735 Å (con 4 qubits y Jordan-Wigner) y corre VQE empezando con todos los parámetros en cero, es decir, desde Hartree-Fock.

b. Compara los dos ansatz (el del ejercicio 2 y este) mostrando la cantidad de qubits, de parámetros y de CNOTs, la cantidad de veces que el optimizador tuvo que medir la energía (```resultado.nfev```) y el error final en mHa.

c. ¿Qué ventajas y desventajas tiene cada ansatz?

In [ ]:
# Escribe tu código aquí para la parte a

In [ ]:
# Escribe tu código aquí para la parte b

## El optimizador

SciPy tiene muchos optimizadores. Algunos, como ```COBYLA``` o ```Powell```, solo usan los valores de la energía para decidir hacia dónde moverse. Otros, como ```L-BFGS-B```, además estiman el **gradiente**, es decir, cuánto cambia la energía cuando movemos un poquito cada parámetro, y usan esa información para moverse más rápido.

En una simulación exacta, todo funciona bien. Pero en una computadora cuántica real, la energía se calcula con una cantidad finita de mediciones, así que cada valor tiene un pequeño error estadístico. Si medimos dos veces la energía con los mismos parámetros, obtenemos dos números un poco distintos. Un optimizador que necesita detectar cambios muy chicos en la energía puede perderse por completo con ese ruido.

Para simular este efecto, podemos usar el ```Estimator``` del simulador Aer con la opción ```default_precision```. Con ```default_precision``` igual a 0,01, cada valor esperado se calcula con un error de aproximadamente 0,01, como pasaría si midiéramos con unos 10.000 shots.

In [ ]:
# Estimator con ruido de medición, con una semilla para que los resultados se puedan repetir
estimator_ruidoso = AerEstimator(options={"default_precision": 0.01, "run_options": {"seed": 5}})

### Ejercicio 4

a. Usando el ansatz químico de $H_2$ a 0,735 Å, corre VQE con los optimizadores ```COBYLA```, ```Powell``` y ```L-BFGS-B``` usando el ```StatevectorEstimator```. Para cada uno, muestra el error final y la cantidad de mediciones.

b. Repite el inciso anterior usando el ```estimator_ruidoso```, con un máximo de 300 iteraciones (```maxiter=300```). Para que la comparación sea justa, calcula el error final evaluando los parámetros que encontró el optimizador (```resultado.x```) con el ```StatevectorEstimator```, que es exacto.

c. ¿Qué optimizador elegirían para una computadora cuántica real? ¿Por qué?

In [ ]:
# Escribe tu código aquí para la parte a

In [ ]:
# Escribe tu código aquí para la parte b

## La curva completa

Ya tenemos todo para calcular la curva de energía de $H_2$ con VQE, como hicimos en la parte 1 con Hartree-Fock y FCI.

Un truco muy útil cuando calculamos una curva es el siguiente. Las moléculas con distancias parecidas tienen estados fundamentales parecidos, así que los parámetros óptimos de una distancia son un muy buen punto de partida para la distancia siguiente. Con esto, el optimizador necesita muchas menos mediciones.

### Ejercicio 5

a. Calcula la energía de $H_2$ con VQE (usando el ansatz químico) para 15 distancias entre 0,3 Å y 3 Å. Para cada distancia, guarda también la energía de Hartree-Fock y la energía exacta. Usen los parámetros óptimos de cada distancia como punto de partida para la siguiente.

b. Grafica las tres curvas.

c. Grafica el error de VQE y el de Hartree-Fock en mHa, con escala logarítmica en el eje vertical (```plt.yscale("log")```), junto con una línea horizontal en la precisión química. ¿VQE logra precisión química en toda la curva?

In [ ]:
# Escribe tu código aquí para la parte a

In [ ]:
# Escribe tu código aquí para la parte b

In [ ]:
# Escribe tu código aquí para la parte c

## Eligiendo con criterio

Con $H_2$ todo funciona. Pero en moléculas más grandes aparecen los problemas de verdad, y hay que tomar decisiones. Una computadora cuántica real tiene pocos qubits, cada CNOT agrega ruido y cada medición de la energía lleva tiempo.

En el siguiente ejercicio, más abierto que los anteriores, van a comparar distintas opciones con la cadena $H_4$ que vimos en la parte 1. Algunas de las combinaciones tardan un minuto en correr, así que tengan paciencia.

### Ejercicio 6

Tomen la cadena $H_4$ con los átomos separados por 1 Å, ```"H 0 0 0; H 0 0 1; H 0 0 2; H 0 0 3"```. Prueben, al menos, estas combinaciones.

- Jordan-Wigner (8 qubits) con el ansatz químico, empezando desde Hartree-Fock.
- Paridad reducida (6 qubits) con ```efficient_su2``` y ```reps``` igual a 1, 2 y 3, empezando con parámetros chicos y aleatorios y con ```maxiter=5000```.

Para cada combinación, muestren la cantidad de qubits, de parámetros, de CNOTs y de grupos de medición del Hamiltoniano, la cantidad de mediciones de la energía que necesitó el optimizador y el error final en mHa.

¿Alguna combinación logra precisión química? Si tuvieran que correr este cálculo en una computadora cuántica real, ¿cuál elegirían y por qué? ¿Se les ocurre alguna otra combinación que valga la pena probar?

In [ ]:
# Escribe tu código aquí

# Muestrear y diagonalizar

## Los problemas de VQE

El ejercicio anterior muestra los problemas que tiene VQE en la práctica. Los ansatz químicos tienen circuitos muy largos, y los ansatz genéricos son difíciles de optimizar. Además, el optimizador necesita medir la energía cientos o miles de veces, y cada una de esas mediciones necesita miles de shots para cada grupo de términos del Hamiltoniano. Con el ruido de las computadoras cuánticas actuales, todo esto se vuelve muy difícil para moléculas interesantes.

## Una idea diferente

En la parte 1 vimos que el estado exacto de una molécula es una superposición de configuraciones, pero que en general casi todo el peso está concentrado en unas pocas. Si supiéramos cuáles son las configuraciones importantes, podríamos armar la matriz del Hamiltoniano usando solamente esas configuraciones y diagonalizarla en una computadora clásica. Esa matriz sería mucho más chica que la matriz completa.

Para encontrar las configuraciones importantes podemos usar la computadora cuántica. Si preparamos un estado que se parezca al estado fundamental y lo medimos muchas veces con el ```Sampler```, las configuraciones con más peso van a aparecer más seguido. No hace falta que el circuito sea perfecto, alcanza con que "apunte" a las configuraciones correctas.

<img src="images/quimica-muestrear-diagonalizar.png" width="700" />

Esta idea es la base de un algoritmo llamado **SQD** (por *sample-based quantum diagonalization*, o "diagonalización cuántica basada en muestras"). Investigadores de IBM y otras instituciones lo usaron en 2024 para estudiar moléculas con hasta 77 qubits en computadoras cuánticas reales, mucho más allá de lo que se puede diagonalizar de forma exacta.

El principio variacional sigue valiendo. Si elegimos un conjunto de $k$ configuraciones $\left|x_1\right>, \ldots, \left|x_k\right>$ y armamos la matriz de $k \times k$ con los elementos $\left<x_i\right|\hat{H}\left|x_j\right>$, su autovalor más chico es la energía del mejor estado que se puede armar combinando solo esas configuraciones. Por lo tanto, es una cota superior de la energía exacta, y se acerca a ella a medida que agregamos las configuraciones correctas.

La siguiente celda define dos funciones. ```energia_en_subespacio``` arma la matriz chica a partir de una lista de bitstrings y devuelve su autovalor más chico. ```es_valida``` dice si un bitstring tiene la cantidad correcta de electrones de cada spin.

Para que sea simple, ```energia_en_subespacio``` arma primero la matriz completa del Hamiltoniano y después se queda con las filas y columnas que necesita. Eso solo funciona para moléculas chicas. Para moléculas grandes, los elementos de la matriz se calculan directamente a partir de las integrales, sin armar nunca la matriz completa.

In [ ]:
def energia_en_subespacio(H, bitstrings, devolver_estado=False):
    """Diagonaliza H usando solo las configuraciones de la lista de bitstrings."""
    matriz = H.to_matrix(sparse=True)
    indices = sorted({int(b, 2) for b in bitstrings})
    # Nos quedamos con las filas y columnas de esas configuraciones
    chica = matriz[indices][:, indices].toarray()
    autovalores, autovectores = np.linalg.eigh(chica)
    if not devolver_estado:
        return autovalores[0]
    # Armamos el estado completo, con amplitud cero en las configuraciones que no usamos
    estado = np.zeros(2 ** H.num_qubits, dtype=complex)
    estado[indices] = autovectores[:, 0]
    return autovalores[0], estado


def es_valida(bitstring, m):
    """Indica si el bitstring tiene la cantidad correcta de electrones de cada spin."""
    norb = m["norb"]
    n_alfa, n_beta = m["nelec"]
    return bitstring[norb:].count("1") == n_alfa and bitstring[:norb].count("1") == n_beta

Vamos a probar esta idea con la cadena $H_4$ estirada, con los átomos separados por 1,5 Å. Al estirar los enlaces aparece la correlación fuerte que vimos en la parte 1, así que Hartree-Fock tiene un error muy grande.

Como circuito vamos a usar el ansatz químico, pero optimizado con solo 40 pasos de VQE, así que está lejos de ser perfecto. Después lo medimos 1000 veces con el ```Sampler``` del simulador Aer.

In [ ]:
m = molecula("H 0 0 0; H 0 0 1.5; H 0 0 3; H 0 0 4.5")
H = hamiltoniano_qubits(m)
print("Error de Hartree-Fock =", 1000 * (m["e_hf"] - m["e_fci"]), "mHa")

# Un circuito aproximado, con solo 40 pasos de VQE
ansatz = ansatz_ucc(m)
resultado, historial = vqe(ansatz, H, maxiter=40)
print("Error del VQE corto =", 1000 * (resultado.fun - m["e_fci"]), "mHa")

# Fijamos los parámetros y agregamos las mediciones
circuito = ansatz.assign_parameters(resultado.x)
circuito.measure_all()

sampler = AerSampler(seed=11)
cuentas = sampler.run([circuito], shots=1000).result()[0].data.meas.get_counts()

validas = [b for b in cuentas if es_valida(b, m)]
energia = energia_en_subespacio(H, validas)
print("Configuraciones distintas =", len(validas))
print("Error de muestrear y diagonalizar =", 1000 * (energia - m["e_fci"]), "mHa")

Con muestras de un circuito que tiene un error de más de 80 mHa, obtuvimos una energía muchísimo mejor. La computadora cuántica no necesitó encontrar el estado exacto, solo tuvo que señalar las configuraciones importantes.

### Ejercicio 7

a. Usando el mismo circuito, calcula la energía con muestrear y diagonalizar para 10, 30, 100, 300, 1000 y 3000 shots. Para cada caso, guarda la cantidad de configuraciones válidas distintas y el error en mHa.

b. Grafica el error en función de la cantidad de shots, con escala logarítmica en el eje horizontal (```plt.xscale("log")```), junto con una línea horizontal en la precisión química y otra en el error del VQE corto.

c. ¿La energía queda alguna vez por debajo de la energía exacta? ¿Por qué?

In [ ]:
# Escribe tu código aquí para la parte a

In [ ]:
# Escribe tu código aquí para la parte b

## Ruido en las muestras

En una computadora cuántica real, algunos qubits se miden mal. Un 0 se lee como 1 o al revés, y entonces aparecen bitstrings con una cantidad incorrecta de electrones, que no son configuraciones válidas de nuestra molécula.

La solución más simple es descartar esos bitstrings, que es lo que venimos haciendo con la función ```es_valida```. Pero a medida que la molécula crece, la fracción de bitstrings que sobreviven sin errores se vuelve muy chica. Por eso, los métodos que se usan en la práctica hacen algo más inteligente, llamado **recuperación de configuraciones**. En lugar de descartar los bitstrings con errores, los corrigen usando información sobre qué tan ocupado está, en promedio, cada orbital.

### Ejercicio 8

a. Escribe una función ```agregar_ruido(bitstrings, p, rng)``` que reciba una lista de bitstrings y devuelva una lista nueva donde cada bit se cambió (de 0 a 1 o de 1 a 0) con probabilidad ```p```. Usen ```rng = np.random.default_rng(0)``` para generar números aleatorios con ```rng.random()```.

b. Toma 1000 muestras del circuito con ```sampler.run([circuito], shots=1000).result()[0].data.meas.get_bitstrings()```, que devuelve la lista de todos los bitstrings medidos. Para $p$ igual a 0, 0,02, 0,05 y 0,1, agrega ruido y muestra la fracción de bitstrings válidos y el error de la energía usando solo los válidos.

c. La probabilidad de que ningún bit de un bitstring cambie es $(1-p)^n$, donde $n$ es la cantidad de qubits. Calcula esa probabilidad para $p = 0{,}02$ y $n$ igual a 8, 50 y 100. ¿Qué les dice eso sobre descartar los bitstrings inválidos en moléculas grandes?

In [ ]:
# Escribe tu código aquí para la parte a

In [ ]:
# Escribe tu código aquí para la parte b

In [ ]:
# Escribe tu código aquí para la parte c

## El truco de combinar α y β

Hay un truco muy simple que agranda el subespacio sin necesitar más muestras. Cada configuración tiene una parte $\alpha$ (la mitad derecha del bitstring) y una parte $\beta$ (la mitad izquierda). Si en una muestra apareció la parte $\alpha$ ```0011``` y en otra muestra apareció la parte $\beta$ ```0101```, es muy probable que la configuración que las combina, ```01010011```, también sea importante, aunque no la hayamos medido.

Entonces, en lugar de usar solo las configuraciones medidas, podemos armar todas las combinaciones posibles entre las partes $\alpha$ y las partes $\beta$ que aparecieron. Esto agranda mucho el subespacio sin costo cuántico extra, y es lo que hace SQD en la práctica.

### Ejercicio 9

a. Escribe una función ```combinar_alfa_beta(bitstrings, m)``` que separe las partes $\alpha$ y $\beta$ de una lista de configuraciones válidas y devuelva todas las combinaciones posibles. Recuerden que la parte $\alpha$ son los últimos ```m["norb"]``` caracteres del bitstring.

b. Para 30, 100 y 300 shots, compara la cantidad de configuraciones y el error usando solo las configuraciones medidas y usando todas las combinaciones.

c. Para $H_4$ hay solo $\binom{4}{2} = 6$ partes $\alpha$ posibles y 6 partes $\beta$ posibles. ¿Qué pasa cuando aparecen todas? ¿Pasaría lo mismo con una molécula grande?

In [ ]:
# Escribe tu código aquí para la parte a

In [ ]:
# Escribe tu código aquí para la parte b

# ¿Cómo sabemos si el resultado está bien?

Hasta ahora, siempre comparamos nuestros resultados con la energía exacta de FCI. Pero la idea de usar una computadora cuántica es estudiar moléculas donde FCI es imposible de calcular. En ese caso, ¿cómo sabemos si podemos confiar en un resultado?

No hay una única respuesta, y en la práctica se combinan varias pruebas. Estas son algunas de las más usadas.

**La cota variacional.** Las energías de VQE y de muestrear y diagonalizar son cotas superiores de la energía exacta. Si dos métodos variacionales dan energías distintas, el que da la energía más baja está más cerca de la respuesta. Hay que tener cuidado, porque no todos los métodos son variacionales. Por ejemplo, **CCSD** (*coupled cluster with singles and doubles*), uno de los métodos clásicos más usados en química, a veces da energías por debajo de la exacta.

**La convergencia.** Si aumentamos los recursos (más shots, un subespacio más grande, más capas en el ansatz o una base más grande) y el resultado deja de cambiar, es una buena señal. Si sigue cambiando, todavía no llegamos.

**La varianza de la energía.** Si un estado es exactamente un autovector del Hamiltoniano, cada vez que medimos la energía obtenemos el mismo valor, así que la varianza $\left<\hat{H}^2\right> - \left<\hat{H}\right>^2$ es cero. Cuanto más chica es la varianza, más cerca estamos de un autovector. La varianza también se puede medir en una computadora cuántica, porque $\hat{H}^2$ es otra suma de operadores de Pauli.

**Comparar con otros métodos.** Siempre conviene comparar con métodos clásicos como Hartree-Fock o CCSD, cuando se pueden calcular, y revisar que el estado tenga las propiedades físicas correctas, como la cantidad de electrones.

### Ejercicio 10

Sigan con la cadena $H_4$ estirada.

a. Calcula la energía de CCSD con ```cc.CCSD(m["hf"]).run().e_tot``` y compárala con la energía de Hartree-Fock y con la exacta. ¿Qué tiene de raro el resultado de CCSD?

b. Sin mirar la energía exacta, calcula la energía de muestrear y diagonalizar (combinando $\alpha$ y $\beta$) para 30, 100, 300 y 1000 shots. ¿El resultado converge?

c. Con ```energia_en_subespacio(H, configuraciones, devolver_estado=True)``` pueden obtener también el estado. Calcula la varianza de la energía para el estado de muestrear y diagonalizar con 30 y con 1000 shots, y también para el estado de Hartree-Fock. Pueden usar ```H.to_matrix(sparse=True)``` para obtener la matriz $M$, y calcular $\left<\hat{H}\right> = \psi^\dagger M \psi$ y $\left<\hat{H}^2\right> = (M\psi)^\dagger (M\psi)$. ¿Qué relación encuentran entre la varianza y el error?

In [ ]:
# Escribe tu código aquí para la parte a

In [ ]:
# Escribe tu código aquí para la parte b

In [ ]:
# Escribe tu código aquí para la parte c

# Resumen

En esta segunda parte vimos dos formas de calcular la energía de una molécula con una computadora cuántica. VQE usa un circuito con parámetros y un optimizador clásico, y su resultado depende mucho de la elección del ansatz, del optimizador y del ruido. Muestrear y diagonalizar usa la computadora cuántica solo para encontrar las configuraciones importantes, y deja que una computadora clásica haga el resto. Por último, vimos cómo evaluar un resultado cuando no tenemos la respuesta exacta para comparar.

# Bibliografía complementaria

- La [documentación de qiskit-addon-sqd](https://qiskit.github.io/qiskit-addon-sqd/), la librería de IBM que implementa muestrear y diagonalizar con recuperación de configuraciones, y su [tutorial de química](https://qiskit.github.io/qiskit-addon-sqd/tutorials/01_chemistry_hamiltonian.html).
- La [documentación de ffsim](https://qiskit-community.github.io/ffsim/), una librería para simular circuitos de química de forma eficiente.
- J. Robledo-Moreno y otros, [*Chemistry beyond exact solutions on a quantum-centric supercomputer*](https://arxiv.org/abs/2405.05068), Science Advances (2025). Es uno de los trabajos principales sobre SQD, con los experimentos de 77 qubits.
- S. McArdle y otros, [*Quantum computational chemistry*](https://arxiv.org/abs/1808.10402), Reviews of Modern Physics (2020).
- J. Tilly y otros, [una revisión muy completa de VQE](https://arxiv.org/abs/2111.05176), con muchas variantes del algoritmo y recomendaciones prácticas, Physics Reports (2022).

La sección de muestrear y diagonalizar está inspirada en el material sobre SQD que IBM preparó para el *Qiskit Fall Fest 2026*.